In [2]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_classic.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_ollama import ChatOllama
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.passthrough import RunnablePassthrough
from sentence_transformers import CrossEncoder
import numpy as np

In [3]:
with open('document.txt', encoding='utf-8') as f:
    text = f.read()
    
headers_to_split_on = [
    ("#", "Header")
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on, strip_headers=False)
chunks = markdown_splitter.split_text(text)

In [4]:
embedding_model = HuggingFaceEmbeddings(
    model_name="keepitreal/vietnamese-sbert", 
    model_kwargs={'device': 'cuda', 'local_files_only': True},   
    encode_kwargs={'normalize_embeddings': True}
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 33617.95it/s]


In [5]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model,
    distance_strategy=DistanceStrategy.COSINE
)

In [5]:
llm = ChatOllama(
    model="qwen2.5:3b",
    num_ctx=4096,
    num_predict=2048,
    num_gpu=0,
    temperature=0
)

In [ ]:
reranking_model = CrossEncoder("Qwen/Qwen3-Reranker-0.6B")

In [6]:
def prompt_chat(x, y):
    prompt = ChatPromptTemplate.from_messages([
        SystemMessagePromptTemplate.from_template(x),
        HumanMessagePromptTemplate.from_template(y)
    ])
    return prompt

In [7]:
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 5
vectorstore_retriever = vectorstore.as_retriever(
    search_type = 'similarity',
    search_kwargs = {'k':5}
)
retrievers = EnsembleRetriever(
    retrievers = [bm25_retriever, vectorstore_retriever],
    weights = [0.5, 0.5]
)

In [8]:
system_instruction = (
    "Bạn là một Chatbot tư vấn y tế thông minh và thân thiện.\n"
    "Nhiệm vụ của bạn là đọc thông tin được cung cấp để diễn đạt lại thành câu trả lời tự nhiên cho người dùng về vấn đề họ gặp phải.\n\n"
    
    "QUY TẮC PHẢN HỒI:\n"
    "- Sử dụng ngôn ngữ ân cần, mạch lạc giống một chuyên viên y tế (ví dụ: 'Qua các triệu chứng bạn chia sẻ, hệ thống nhận thấy...').\n"
    "- Không thêm, không bớt, không suy luận, không nói rằng không tìm thấy dữ liệu.\n"
    "- Triển khai mỗi ý nằm ổ một dòng, không đưa câu trả lời lên một dòng duy nhất.\n"
    "- Luôn kết thúc bằng một lời khuyên người dùng nên đến các cơ sở y tế hoặc gặp bác sĩ để thăm khám trực tiếp, vì đây chỉ là thông tin tham khảo từ hệ thống."
)

human_instruction = (
    "Báo cáo kết quả đối chiếu từ hệ thống Python:\n{context}\n\n"
    "Câu hỏi gốc của người dùng: {question}"
)

In [17]:
query = 'tôi bị tiêu chảy, buồn nôn, đau bụng, đầy hơi là bệnh gì'

In [18]:
docs = retrievers.invoke(query)

In [11]:
def format_docs(documents):
    return "\n\n".join(doc.page_content for doc in documents)

In [ ]:
pairs = [(query, doc.page_content) for doc in docs]
score = reranking_model.predict(pairs)

In [14]:
rag_chain = (
    {'context':lambda _: format_docs(docs), 'question': RunnablePassthrough()}
    | prompt_chat(system_instruction, human_instruction)
    | llm
    | StrOutputParser()
)

In [20]:
answer = rag_chain.invoke(docs[np.where(score==np.max(score))[0][0]].page_content)

In [21]:
print(answer)

Qua các triệu chứng bạn chia sẻ, hệ thống nhận thấy Hội chứng quai mù thường có dấu hiệu như: cảm giác ăn không ngon, bị đau bụng, buồn nôn, có cảm giác đầy hơi, bị khó chịu sau ăn, mắc bệnh tiêu chảy và giảm cân không rõ nguyên nhân. Hệ thống khuyến nghị bạn cần nhanh chóng đến cơ sở y tế để được chẩn đoán và điều trị kịp thời, đặc biệt khi có cảm giác buồn nôn, tiêu chảy kéo dài.
